# PyQt5 QTableWidget — Öğrenci Not Takip Sistemi
### Görsel Programlama | Qt Designer → `pyuic` → Python

**Hedef:** Verilen `NotPenceresi` kodunu satır satır anlayarak öğrenci ekleme, numara kontrolü, ortalama hesaplama, satır silme ve tabloyu temizleme işlemlerini öğrenmek.

> Bu notebook açıklama ve alıştırma içindir. Masaüstü GUI kodunu PyCharm'da `.py` dosyası olarak çalıştıracağız. `uic.loadUi()` kullanılmaz.

## 1. Neden QTableWidget?
`QListWidget` tek boyutlu bir liste gösterirken `QTableWidget` satır ve sütunlardan oluşan bir tablo sunar. Her hücreye `QTableWidgetItem` yerleştirilir. Bu uygulamada **5 sütun** vardır: Numara (0), Ad Soyad (1), Vize (2), Final (3), Ortalama (4).

| Satır | 0 — Numara | 1 — Ad Soyad | 2 — Vize | 3 — Final | 4 — Ortalama |
|---|---|---|---|---|---|
| 0 | 2026001 | Ayşe Yılmaz | 80 | 90 | 86.0 |
| 1 | 2026002 | Ali Demir | 60 | 70 | 66.0 |

**Düşün:** Neden ilk satırın indeksi 0? `item(1, 3)` hangi hücreyi ifade eder?

## 2. Qt Designer'da hangi nesneler olmalı?

| Widget türü | `objectName` | İşlev |
|---|---|---|
| QLineEdit | `txtNumara` | Öğrenci numarası |
| QLineEdit | `txtAd` | Ad soyad |
| QSpinBox | `spinVize` | 0–100 arası vize |
| QSpinBox | `spinFinal` | 0–100 arası final |
| QTableWidget | `tableOgrenciler` | Kayıtların gösterimi |
| QPushButton | `btnEkle` | Ekle |
| QPushButton | `btnSil` | Seçileni sil |
| QPushButton | `btnTemizle` | Tüm satırları temizle |
| QLabel | `lblDurum` | Öğrenci sayısı |

Qt Designer'da bir **Main Window** açıp `centralwidget` üzerine bu bileşenleri yerleştirin. `QFormLayout` (alanlar), `QHBoxLayout` (butonlar), `QVBoxLayout` (tüm ekran) kullanabilirsiniz.

**Beklenen görünüm (şematik):**

```text
┌──────────── Öğrenci Not Takip Sistemi ────────────┐
│ Numara: [_________]   Ad Soyad: [______________] │
│ Vize:   [  0 ▲▼ ]    Final:    [  0 ▲▼ ]        │
│ [Öğrenci Ekle] [Seçileni Sil] [Tümünü Temizle]  │
│ ┌────────┬──────────┬──────┬───────┬─────────┐ │
│ │ Numara │ Ad Soyad │ Vize │ Final │ Ortalama│ │
│ ├────────┼──────────┼──────┼───────┼─────────┤ │
│ │ ...    │ ...      │ ...  │ ...   │ ...     │ │
│ └────────┴──────────┴──────┴───────┴─────────┘ │
│ Öğrenci sayısı: 0                                │
└──────────────────────────────────────────────────┘
```

## 3. `.ui` dosyasını Python'a dönüştürmek

PyCharm terminalinde `.ui` dosyasının bulunduğu klasörde çalıştırın:

```bash
python -m PyQt5.uic.pyuic tablo_notlar.ui -o ui_tablo_notlar.py
```

Ardından uygulama dosyası:

```bash
python tablo_notlar.py
```

- `tablo_notlar.ui`: Qt Designer tasarımı.
- `ui_tablo_notlar.py`: **otomatik üretilen** arayüz sınıfı (`Ui_MainWindow`).
- `tablo_notlar.py`: **bizim yazdığımız** olay ve iş mantığı.

> Otomatik üretilen dosyayı elle düzenlemeyin. Designer'da değişiklik yaptıktan sonra komutu yeniden çalıştırın.

## 4. İçe aktarımlar ve sınıf yapısı

`QApplication` olay döngüsünü başlatır. `QMainWindow` ana pencere sınıfıdır. `QTableWidgetItem` tablo hücresi oluşturur. `QMessageBox` uyarı/bilgi kutusu açar. `QAbstractItemView` seçim ve düzenleme davranışları için sabitler sunar.

In [ ]:
import sys
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QTableWidgetItem,
    QMessageBox, QAbstractItemView
)
from ui_tablo_notlar import Ui_MainWindow  # pyuic ile üretilir


## 5. Kalıtım ve `setupUi(self)`

`class NotPenceresi(QMainWindow, Ui_MainWindow)` iki sınıftan kalıtım alır:
- `QMainWindow`: pencerenin temel davranışı.
- `Ui_MainWindow`: Designer'da tanımlanan bileşenleri kuran sınıf.

`super().__init__()` temel pencereyi hazırlar. `self.setupUi(self)` buton, tablo, metin kutuları gibi nesneleri oluşturur. Bu yüzden `self.btnEkle` gibi isimler kullanılabilir.

**Önemli:** `setupUi()` bir `.ui` dosyasını çalışma anında okumaz; önceden üretilen Python kodunu çalıştırır.

In [ ]:
class NotPenceresi(QMainWindow, Ui_MainWindow):
    def __init__(self):
        super().__init__()  # QMainWindow kurucusu
        self.setupUi(self)  # Designer arayüzünü oluştur


## 6. Tablo ayarlarını anlama

- `setColumnCount(5)`: 5 sütun açar.
- `setHorizontalHeaderLabels(...)`: başlıkları tanımlar.
- `SelectRows`: hücre yerine satır seçilir.
- `SingleSelection`: aynı anda tek seçim.
- `NoEditTriggers`: kullanıcı hücreye çift tıklayıp metni değiştiremez; değişiklik kodla yapılır.
- `setRange(0, 100)`: QSpinBox not sınırları.

Bu ayarlar veri eklemez; tablonun **davranışını** belirler.

In [ ]:
self.tableOgrenciler.setColumnCount(5)
self.tableOgrenciler.setHorizontalHeaderLabels(
    ["Numara", "Ad Soyad", "Vize", "Final", "Ortalama"]
)
self.tableOgrenciler.setSelectionBehavior(QAbstractItemView.SelectRows)
self.tableOgrenciler.setSelectionMode(QAbstractItemView.SingleSelection)
self.tableOgrenciler.setEditTriggers(QAbstractItemView.NoEditTriggers)
self.spinVize.setRange(0, 100)
self.spinFinal.setRange(0, 100)

## 7. Signal–Slot: Butona basınca ne olur?

`clicked` bir **signal**, `ekle`, `sil`, `temizle` ise çağrılan metotlardır. `connect(self.ekle)` yazılır; `connect(self.ekle())` yazılırsa fonksiyon bağlantı anında çağrılır.

In [ ]:
self.btnEkle.clicked.connect(self.ekle)
self.btnSil.clicked.connect(self.sil)
self.btnTemizle.clicked.connect(self.temizle)

## 8. Öğrenci sayısını güncelleme

`rowCount()` tablodaki **veri satırı** sayısını döndürür. Başlık satırını saymaz. `lblDurum.setText()` etikette görünen metni değiştirir.

In [ ]:
def durum_guncelle(self):
    self.lblDurum.setText(
        f"Öğrenci sayısı: {self.tableOgrenciler.rowCount()}"
    )

## 9. Ekleme — Adım 1: Alanları okuma ve doğrulama

`text()` QLineEdit'teki metni döndürür. `strip()` baştaki/sondaki boşlukları kaldırır. `not no or not ad` ifadesi alanlardan biri boşsa doğrudur. `return` işlemi durdurur.

In [ ]:
no = self.txtNumara.text().strip()
ad = self.txtAd.text().strip()
if not no or not ad:
    QMessageBox.warning(self, "Eksik bilgi", "Numara ve ad soyad giriniz.")
    return

## 10. Ekleme — Adım 2: Aynı numara daha önce kaydedilmiş mi?

`range(rowCount())` ile tüm satırlar gezilir. `item(satir, 0)` o satırın **Numara** sütunundaki `QTableWidgetItem` nesnesidir; `.text()` ile numara okunur.

Bu kodda hücrelerin `ekle()` ile doldurulmuş olduğu varsayılır. Başka bir kod boş satır eklerse `item()` `None` dönebilir; bu durumda kontrol eklenmelidir.

In [ ]:
for satir in range(self.tableOgrenciler.rowCount()):
    if self.tableOgrenciler.item(satir, 0).text() == no:
        QMessageBox.warning(self, "Tekrar", "Bu numara zaten kayıtlı.")
        return

## 11. Ekleme — Adım 3: Ortalama hesaplama

Vize ağırlığı %40, final ağırlığı %60'tır. `value()` QSpinBox'tan sayısal değer döndürür.

In [ ]:
vize, final = self.spinVize.value(), self.spinFinal.value()
ortalama = vize * 0.40 + final * 0.60
print(80 * 0.40 + 90 * 0.60)  # 86.0

## 12. Ekleme — Adım 4: Yeni satır ve hücre oluşturma

`rowCount()` mevcut satır sayısını verir; yeni satırın indeksi olarak kullanılır. `insertRow(satir)` boş satır açar. `enumerate()` hem sütun numarasını hem değeri verir. `QTableWidgetItem` hücre içeriğini temsil eder.

`f"{ortalama:.1f}"` ortalamayı bir ondalık basamakla gösterir.

In [ ]:
satir = self.tableOgrenciler.rowCount()
self.tableOgrenciler.insertRow(satir)
for sutun, deger in enumerate(
    [no, ad, str(vize), str(final), f"{ortalama:.1f}"]
):
    self.tableOgrenciler.setItem(satir, sutun, QTableWidgetItem(deger))

## 13. Ekleme — Adım 5: Formu temizleme

`clear()` metin alanlarını temizler. Verilen özgün kod, QSpinBox değerlerini sıfırlamaz; yalnızca numara ve ad alanlarını temizler. `durum_guncelle()` sayacı yeniler.

In [ ]:
self.txtNumara.clear()
self.txtAd.clear()
self.durum_guncelle()

## 14. Satır silme

`currentRow()` seçili satırın indeksini verir. Seçim yoksa `-1` döner. `removeRow(satir)` satırı kaldırır. İşlemden sonra öğrenci sayısı güncellenir.

In [ ]:
def sil(self):
    satir = self.tableOgrenciler.currentRow()
    if satir < 0:
        QMessageBox.information(self, "Seçim", "Silmek için bir satır seçiniz.")
        return
    self.tableOgrenciler.removeRow(satir)
    self.durum_guncelle()

## 15. Tüm satırları temizleme

`setRowCount(0)` satır sayısını sıfıra indirir; **sütun başlıkları korunur**. `clear()` ise tablo öğeleriyle birlikte başlık öğelerini de temizleyebilir. Bu uygulamada `setRowCount(0)` uygundur.

In [ ]:
def temizle(self):
    self.tableOgrenciler.setRowCount(0)
    self.durum_guncelle()

## 16. Uygulamanın tamamı — açıklamalı sürüm

Aşağıdaki kodu `tablo_notlar.py` dosyasına kopyalayın. Aynı klasörde dönüştürülmüş `ui_tablo_notlar.py` bulunmalıdır.

In [ ]:
"""QTableWidget ile Öğrenci Not Takip Sistemi.
Önce: python -m PyQt5.uic.pyuic tablo_notlar.ui -o ui_tablo_notlar.py
Sonra: python tablo_notlar.py
"""
import sys
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QTableWidgetItem,
    QMessageBox, QAbstractItemView
)
from ui_tablo_notlar import Ui_MainWindow

class NotPenceresi(QMainWindow, Ui_MainWindow):
    def __init__(self):
        super().__init__()             # Ana pencereyi hazırla
        self.setupUi(self)             # Üretilmiş arayüzü kur; loadUi kullanılmaz

        # Tablo 5 sütundan oluşur; indeksler 0'dan başlar.
        self.tableOgrenciler.setColumnCount(5)
        self.tableOgrenciler.setHorizontalHeaderLabels(
            ["Numara", "Ad Soyad", "Vize", "Final", "Ortalama"]
        )
        # Kullanıcı tek bir satır seçebilir; hücreleri doğrudan değiştiremez.
        self.tableOgrenciler.setSelectionBehavior(QAbstractItemView.SelectRows)
        self.tableOgrenciler.setSelectionMode(QAbstractItemView.SingleSelection)
        self.tableOgrenciler.setEditTriggers(QAbstractItemView.NoEditTriggers)
        self.spinVize.setRange(0, 100)
        self.spinFinal.setRange(0, 100)

        # Sinyalleri sınıf metotlarına bağla.
        self.btnEkle.clicked.connect(self.ekle)
        self.btnSil.clicked.connect(self.sil)
        self.btnTemizle.clicked.connect(self.temizle)
        self.durum_guncelle()  # İlk açılışta öğrenci sayısı: 0

    def durum_guncelle(self):
        # Tablodaki satır sayısını kullanıcıya göster.
        self.lblDurum.setText(
            f"Öğrenci sayısı: {self.tableOgrenciler.rowCount()}"
        )

    def ekle(self):
        # Metin alanlarını oku; gereksiz boşlukları sil.
        no = self.txtNumara.text().strip()
        ad = self.txtAd.text().strip()
        if not no or not ad:
            QMessageBox.warning(self, "Eksik bilgi", "Numara ve ad soyad giriniz.")
            return

        # Aynı öğrenci numarasının ikinci kez eklenmesini engelle.
        for satir in range(self.tableOgrenciler.rowCount()):
            if self.tableOgrenciler.item(satir, 0).text() == no:
                QMessageBox.warning(self, "Tekrar", "Bu numara zaten kayıtlı.")
                return

        # Notları al ve ağırlıklı ortalamayı hesapla.
        vize = self.spinVize.value()
        final = self.spinFinal.value()
        ortalama = vize * 0.40 + final * 0.60

        # Tablonun sonuna yeni satır ekle.
        satir = self.tableOgrenciler.rowCount()
        self.tableOgrenciler.insertRow(satir)

        # Beş hücreye sırayla metin yerleştir.
        veriler = [no, ad, str(vize), str(final), f"{ortalama:.1f}"]
        for sutun, deger in enumerate(veriler):
            self.tableOgrenciler.setItem(satir, sutun, QTableWidgetItem(deger))

        # Yeni kayıt için metin kutularını temizle ve sayacı güncelle.
        self.txtNumara.clear()
        self.txtAd.clear()
        self.durum_guncelle()

    def sil(self):
        # Seçili satır yoksa currentRow() -1 verir.
        satir = self.tableOgrenciler.currentRow()
        if satir < 0:
            QMessageBox.information(self, "Seçim", "Silmek için bir satır seçiniz.")
            return
        self.tableOgrenciler.removeRow(satir)
        self.durum_guncelle()

    def temizle(self):
        # Sütun başlıklarını koruyarak tüm öğrenci satırlarını kaldır.
        self.tableOgrenciler.setRowCount(0)
        self.durum_guncelle()

if __name__ == "__main__":
    app = QApplication(sys.argv)  # Qt uygulamasını oluştur
    pencere = NotPenceresi()       # Pencere nesnesi
    pencere.show()                 # Pencereyi göster
    sys.exit(app.exec_())         # Olay döngüsünü çalıştır


## 17. Olay akışını takip edelim

```text
Kullanıcı "Öğrenci Ekle" butonuna basar
                ↓
       clicked → ekle()
                ↓
     Numara ve ad dolu mu? ──Hayır──→ Uyarı
                │ Evet
                ↓
    Numara tabloda mevcut mu? ──Evet──→ Uyarı
                │ Hayır
                ↓
       Vize ve finali oku
                ↓
         Ortalama hesapla
                ↓
    insertRow + setItem × 5
                ↓
      Alanları ve sayacı güncelle
```

## 18. Sıra Sizde 1 — Formu tamamen sıfırlama

**Görev:** Yeni öğrenci eklendikten sonra `spinVize` ve `spinFinal` değerleri de sıfırlansın. Nereye kod eklemeniz gerekir?

<details><summary>Çözümü Göster</summary>

`ekle()` metodunun sonunda, `txtAd.clear()` sonrasına:

```python
self.spinVize.setValue(0)
self.spinFinal.setValue(0)
```

</details>

## 19. Sıra Sizde 2 — Başarı durumunu ekleme

**Görev:** Ortalaması 60 veya üzeri olan öğrencinin başarılı sayıldığını kabul edin. Altıncı sütun ekleyip `Başarılı` veya `Başarısız` yazdırın.

<details><summary>Çözümü Göster</summary>

`__init__` içinde:

```python
self.tableOgrenciler.setColumnCount(6)
self.tableOgrenciler.setHorizontalHeaderLabels(
    ["Numara", "Ad Soyad", "Vize", "Final", "Ortalama", "Durum"]
)
```

`ekle()` içinde ortalama hesaplandıktan sonra:

```python
durum = "Başarılı" if ortalama >= 60 else "Başarısız"
veriler = [no, ad, str(vize), str(final), f"{ortalama:.1f}", durum]
for sutun, deger in enumerate(veriler):
    self.tableOgrenciler.setItem(satir, sutun, QTableWidgetItem(deger))
```

Bu değişiklikte özgün 5 elemanlı döngü yerine 6 elemanlı döngü kullanılmalıdır.

</details>

## 20. Sıra Sizde 3 — Silmeden önce onay isteme

**Görev:** Kullanıcı `Sil` butonuna basınca `Evet/Hayır` onayı gösterilsin. `Hayır` derse kayıt silinmesin.

<details><summary>Çözümü Göster</summary>

`sil()` içinde `removeRow()` öncesine:

```python
cevap = QMessageBox.question(
    self, "Silme onayı", "Seçili öğrenciyi silmek istiyor musunuz?",
    QMessageBox.Yes | QMessageBox.No,
    QMessageBox.No
)
if cevap != QMessageBox.Yes:
    return
```

</details>

## 21. Sıra Sizde 4 — Seçili öğrenciyi forma getir

**Görev:** Bir satıra tıklanınca numara, ad, vize ve final bilgileri forma aktarılsın. `cellClicked` sinyalini araştırın.

<details><summary>Çözümü Göster</summary>

`__init__` içine:

```python
self.tableOgrenciler.cellClicked.connect(self.satiri_forma_aktar)
```

Sınıfa yeni metot:

```python
def satiri_forma_aktar(self, satir, sutun):
    self.txtNumara.setText(self.tableOgrenciler.item(satir, 0).text())
    self.txtAd.setText(self.tableOgrenciler.item(satir, 1).text())
    self.spinVize.setValue(int(self.tableOgrenciler.item(satir, 2).text()))
    self.spinFinal.setValue(int(self.tableOgrenciler.item(satir, 3).text()))
```

Bu yalnızca bilgileri forma getirir; **güncelleme/kaydetme işlemi eklemez**.

</details>

## 22. Hata bulma etkinliği

Aşağıdaki satırlarda ne yanlış?

```python
self.btnEkle.clicked.connect(self.ekle())
self.tableOgrenciler.setItem(satir, 5, QTableWidgetItem("86"))
self.tableOgrenciler.removeRow(-1)
```

<details><summary>Çözümü Göster</summary>

1. `connect(self.ekle)` olmalı; parantez fonksiyonu hemen çalıştırır.
2. 5 sütunlu tabloda geçerli indeksler 0–4'tür; ortalama sütunu `4`.
3. Seçili satır `-1` ise silme yapılmamalı; önce `currentRow() >= 0` kontrol edilmeli.

</details>

## 23. Mini değerlendirme

1. `QTableWidgetItem` neden kullanılır?
2. `setRowCount(0)` ile `clear()` arasındaki fark nedir?
3. `currentRow()` ne zaman `-1` verir?
4. `enumerate()` neden işimize yarıyor?
5. `setupUi(self)` ile `uic.loadUi()` arasındaki fark nedir?
6. Aynı numarayı kontrol etmeseydik ne olurdu?
7. Ortalama hesabında `0.40` ve `0.60` neyi ifade eder?

<details><summary>Kısa cevapları göster</summary>

1. Hücrede gösterilecek öğeyi oluşturur. 2. İlk yöntem satırları kaldırıp sütun başlıklarını korur. 3. Seçim olmadığında. 4. Sütun indeksiyle veriyi birlikte sağlar. 5. Biri önceden üretilmiş Python sınıfını kurar, diğeri çalışma anında `.ui` okur. 6. Aynı öğrenci iki kez kaydedilebilirdi. 7. Vize ve final ağırlıkları.

</details>

## 24. Öğretmen için ders akışı (yaklaşık 3 saat)

| Süre | Etkinlik |
|---|---|
| 20 dk | QListWidget–QTableWidget farkı, satır/sütun kavramı |
| 25 dk | Designer'da form ve tablo tasarımı |
| 15 dk | `.ui` → `.py`, kalıtım, `setupUi` |
| 25 dk | Tablo ayarları ve Signal–Slot |
| 35 dk | Ekleme: doğrulama, tekrar kontrolü, ortalama |
| 20 dk | Satır silme, temizleme, sayaç |
| 25 dk | Sıra Sizde 1–3 |
| 15 dk | Hata bulma ve mini değerlendirme |

**Not:** Bu uygulama yalnızca bellek üzerinde çalışır. Kapatıldığında kayıtlar kaybolur. Veritabanı/dosya kaydı sonraki ders konusu olabilir.